# 🧪 PT-W1-D4 概念实验：语义三层缺失验证

> 配套阅读：同名 .md
> 实验目标：模拟 AI 读 Domain Model 的困境——有结构无语义

## 第 1 格：Domain Model 有什么 vs AI 能回答什么

In [ ]:
domain_model_relations = {
    ("Contract", "Space"):     {"fk": "shop_id", "rel_type": "Identity Reference"},
    ("Contract", "Merchant"):   {"fk": "tenant_id", "rel_type": "Identity Reference"},
    ("Contract", "ContractClause"): {"fk": "contract_id", "rel_type": "Structural Composition"},
    ("Project", "Building"):   {"fk": "project_id", "rel_type": "Hierarchical Containment"},
    ("Building", "Floor"):     {"fk": "building_id", "rel_type": "Hierarchical Containment"},
}

print("AI 从 Domain Model 能看到的：")
for (a, b), info in domain_model_relations.items():
    print(f"  {a}.{info['fk']} → FK → {b}  [{info['rel_type']}]")

unanswerable = [
    "合同终止后铺位多久可以重新出租？",
    "商户逾期账单 > 3 笔时新合同需谁审批？",
    "Lease 过期但 Inspection 未完成，铺位真实状态？",
]
print("\nAI 无法回答的问题：")
for q in unanswerable:
    print(f"  ❌ {q}")

## 第 2 格：缺失 1 — 语义命名（Semantic Naming）

In [ ]:
fk_view = {"Contract→Space": "Identity Reference"}
semantic_view = {"Contract→Space": "Contract **occupies** Space"}

def ai_can_reason(view_type):
    if view_type == "FK":
        return "只知道'有关系'，不能推理方向和后果"
    else:
        return "能推理：合同终止 → 应该 release Space"

for pair, desc in fk_view.items():
    print(f"FK 视角:   {pair} = {desc}")
    print(f"  → AI: {ai_can_reason('FK')}")
for pair, desc in semantic_view.items():
    print(f"\n语义视角:  {pair} = {desc}")
    print(f"  → AI: {ai_can_reason('Semantic')}")

## 第 3 格：缺失 2 & 3 — 业务规则和推理链

In [ ]:
def naive_answer():
    return "根据 shop.status='occupied'，该铺位被占用。"

def ontology_answer(facts):
    steps = []
    steps.append("1. Entity: Space A101 (ResourceUnit)")
    steps.append("2. Relationship: A101 --occupies--> Lease #2023-085")
    lease_status = facts.get("lease_status")
    steps.append(f"3. Lifecycle: Lease 状态 = {lease_status}")
    if lease_status == "Terminating":
        inspection = facts.get("inspection_done")
        steps.append("4. Rule: 退租流程未完成 → Space 不可出租")
        if not inspection:
            steps.append("5. Inspection 未完成 → 结论：不可出租，需先完成验收")
        else:
            steps.append("5. Inspection 已完成 → 结论：可释放进入可招商清单")
    return steps

print("【无 Ontology】Agent 回答：")
print(f"  {naive_answer()}")
print("\n【有 Ontology】Agent 推理链：")
for step in ontology_answer({"lease_status": "Terminating", "inspection_done": False}):
    print(f"  {step}")

## 第 4 格：三轴分离 — Ontology 的三个认知层次

In [ ]:
from matplotlib import font_manager, pyplot as plt
import numpy as np

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print("字体:", font_name)
layers = ["① 静态拓扑\n(有什么/怎么连接)", "② 动态演化\n(怎么变化/如何传播)", "③ 能力编排\n(AI能做什么)"]
coverage = [0.7, 0.3, 0.1]

fig, ax = plt.subplots(figsize=(8, 4))
colors_layer = ["#2ecc71", "#f39c12", "#e74c3c"]
ax.barh(range(3), coverage, color=colors_layer, alpha=0.8, height=0.6)
ax.set_yticks(range(3))
ax.set_yticklabels(layers, fontsize=11)
ax.set_xlabel("Domain Model 覆盖度", fontsize=12)
ax.set_title("Ontology 三层认知 vs Domain Model 覆盖", fontsize=14)
ax.set_xlim(0, 1.1)
for i, v in enumerate(coverage):
    ax.text(v + 0.02, i, f"{v*100:.0f}%", va="center", fontsize=11)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig("/tmp/w1d4_three_axes.png", dpi=120)
plt.show()
print("Domain Model 是地基（静态拓扑），Semantic Model 是建筑（三层都要覆盖）")